## IMPORTS


In [1]:
import pandas as pd
import numpy as np
import optuna
from sklearn.impute import SimpleImputer
import os
from imblearn.pipeline import Pipeline 
import xgboost as  xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import sys
from sklearn.experimental import enable_iterative_imputer
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.combine import SMOTETomek, SMOTEENN
from sklearn.impute import IterativeImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
import joblib
import logging
import optuna

ruta_raiz = os.path.abspath('..')
if ruta_raiz not in sys.path:
    sys.path.append(ruta_raiz)

%load_ext autoreload
%autoreload 2
from Data_preprocesing.IQRCapper import IQRCapper



c:\Users\gonzalo.iglesias\AppData\Local\miniconda3\envs\machinelearning2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LOS OUTPUTS DE OPTUNA QUE LOS IMPRIMA EN UN TXT PARA SIN VOLVERLO A CORRER

In [2]:
log_file = logging.FileHandler("optuna_resultados.txt")
log_file.setFormatter(logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s'))
optuna.logging.get_logger("optuna").addHandler(log_file)

## LOAD DATA + SAVE FUCTION


In [3]:
X_train = pd.read_csv("../Train_test_split/X_train.csv")
y_train = pd.read_csv("../Train_test_split/y_train.csv")
X_test = pd.read_csv("../Train_test_split/X_test.csv")
y_test = pd.read_csv("../Train_test_split/y_test.csv")



target_folder = '../comparations'

def save_experiment(model_name, imbalance_method, accuracy, precision, recall, f1, roc_auc):
    new_result = {
        'Model': [model_name],
        'Imbalance_Method': [imbalance_method],
        'Accuracy': [round(accuracy, 4)],
        'Precision': [round(precision, 4)],
        'Recall': [round(recall, 4)],
        'F1_Score': [round(f1, 4)],
        'ROC_AUC': [round(roc_auc, 4)]
    }
    df_new = pd.DataFrame(new_result)
    
    # Point the file to inside the folder
    csv_file = f'{target_folder}/imbalance_results.csv'
    
    if os.path.exists(csv_file):
        df_new.to_csv(csv_file, mode='a', header=False, index=False)
    else:
        df_new.to_csv(csv_file, index=False)
        
    print(f"✅ Success! Results for {model_name} + {imbalance_method} saved in the '{target_folder}' folder.")


## XGBOOST

In [4]:
# Vamos a dejar los splits en el formato necesario
train = xgb.DMatrix(data=X_train, label=y_train)
test  = xgb.DMatrix(data=X_test, label=y_test)

#vamos a definir la variable objetivo
num_classes = int(y_train['customer_segment'].nunique())

learning_objective = {
    'objective': 'multi:softmax', 
    'num_class': num_classes,
    'eval_metric': 'mlogloss'
}

# Creamos un modelo "basico para confirmar que el modelo es util para nuestro caso"
model = xgb.train(params = learning_objective, dtrain= train)
test_predictions = model.predict(test)
print(accuracy_score(y_test, test_predictions))
print(f1_score(y_test, test_predictions, average='weighted'))


0.7569
0.7536337642063996


Con un modelo basico sin ningun tipo de ajuste de hyperparametros obtenemos un accuracy de 0.76 y un f1 score de 0.75(buena señal). Viedno los resultados decidimos seguir con el siguiente paso: Optuna (el metedo de balanceo lo decidiremos como si fuse un hyperparametro de optuna)

In [5]:
modelo_xgb = xgb.XGBClassifier(
    objective='multi:softmax', 
    num_class=num_classes, 
    random_state=42,
    eval_metric='mlogloss'
)

## DEFINIMOS LOS RANGOS Y CATEGORIAS QUE QUEREMOS QUE OPTUNA OPTIMICE

In [6]:

  
def objective(trial):
    imbalance_methods = {
        "Baseline": None,
        "RandomUnderSampler":     RandomUnderSampler(random_state=42),
        "TomekLinks":             TomekLinks(),
        "RandomOverSampler":      RandomOverSampler(random_state=42),
        "SMOTE":                  SMOTE(random_state=42),
        "ADASYN":                 ADASYN(random_state=42),
        "SMOTETomek":             SMOTETomek(random_state=42),
        "SMOTEENN":               SMOTEENN(random_state=42),
    }
    # ESCOGER TIPO DE MUESTREO
    nombre_metodo = trial.suggest_categorical('imbalance_method', list(imbalance_methods.keys()))
    metodo_elegido = imbalance_methods[nombre_metodo]

    # ESCOGER LOS HIPERPARÁMETROS DE XGBOOST
    param_n_estimators = trial.suggest_int('n_estimators', 50, 1000)   
    param_max_depth = trial.suggest_int('max_depth', 3, 12)
    param_learning_rate = trial.suggest_float('learning_rate', 0.005, 0.3, log=True)
    param_gamma = trial.suggest_float('gamma', 1e-8, 1.0, log=True)
    param_lambda = trial.suggest_float('lambda', 1e-8, 10.0, log=True)
    param_alpha = trial.suggest_float('alpha', 1e-8, 10.0, log=True)

    param_subsample = trial.suggest_float('subsample', 0.5, 1.0)
    param_colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    modelo_xgb = xgb.XGBClassifier(
        n_estimators=param_n_estimators,
        max_depth=param_max_depth,
        learning_rate=param_learning_rate,
        subsample=param_subsample,
        colsample_bytree=param_colsample_bytree,
        objective='multi:softmax',
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1 
    )

    pasos_pipeline = [
        ('capping_outliers', IQRCapper(factor=1.5)),
        ('imputacion', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()), 
    ]
    
    if metodo_elegido is not None: #Si no se ha elegido imbalance_methods Baseline
        pasos_pipeline.append(('sampler', metodo_elegido))
        
    pasos_pipeline.append(('clf', modelo_xgb))
    
    pipeline_completo = Pipeline(pasos_pipeline)

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42) 
    scores = cross_val_score(
        pipeline_completo, 
        X_train, y_train, 
        cv=cv, 
        scoring='f1_macro', 
        n_jobs=1
    )
    
    return np.mean(scores)
            
  

EJECUTAMOS OPTUNA

In [7]:
study = optuna.create_study(direction='maximize', study_name="Optimizacion_XGB_Balanceo")
study.optimize(objective, n_trials=None, timeout=28800) #PUEDE ESTAR HASTA 8 HORAS ENTRENANDO (TODA LA NOCHE)
print(f"Mejor F1-Score Macro: {study.best_value:.4f}")
print("Mejor combinación encontrada:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")

with open("mejores_parametros_XGBoost.txt", "w") as archivo:
    archivo.write("Mejores hiperparámetros encontrados para XGBoost:\n")
    archivo.write("="*40 + "\n")
    for key, value in study.best_params.items():
        archivo.write(f"  - {key}: {value}\n")

print("¡Parámetros guardados correctamente en 'mejores_parametros.txt'!")


[I 2026-03-15 22:05:02,498] A new study created in memory with name: Optimizacion_XGB_Balanceo
[I 2026-03-15 22:05:33,117] Trial 0 finished with value: 0.7138394570813714 and parameters: {'imbalance_method': 'SMOTEENN', 'n_estimators': 718, 'max_depth': 4, 'learning_rate': 0.012713028407710558, 'gamma': 1.1699440705459433e-05, 'lambda': 1.1231410286104488e-07, 'alpha': 0.003557309060870807, 'subsample': 0.5443938658136758, 'colsample_bytree': 0.7055625786657869}. Best is trial 0 with value: 0.7138394570813714.
[I 2026-03-15 22:07:03,133] Trial 1 finished with value: 0.7197107003483393 and parameters: {'imbalance_method': 'SMOTEENN', 'n_estimators': 946, 'max_depth': 10, 'learning_rate': 0.017121480638268734, 'gamma': 0.11392698097560597, 'lambda': 3.199003760590812e-05, 'alpha': 1.919788501756336e-08, 'subsample': 0.5492313386028445, 'colsample_bytree': 0.5085238396368866}. Best is trial 1 with value: 0.7197107003483393.
[I 2026-03-15 22:07:21,837] Trial 2 finished with value: 0.748902

Mejor F1-Score Macro: 0.7613
Mejor combinación encontrada:
  - imbalance_method: Baseline
  - n_estimators: 539
  - max_depth: 7
  - learning_rate: 0.0181116545131285
  - gamma: 7.454094627623283e-06
  - lambda: 0.00036780803167629474
  - alpha: 0.014738256092695032
  - subsample: 0.7610581670971341
  - colsample_bytree: 0.7542241550990665
¡Parámetros guardados correctamente en 'mejores_parametros.txt'!


GUARDAMOS EL MEJOR MODELO

In [8]:
print("Reconstruyendo y entrenando el modelo final con toda la data...")

imbalance_methods = {
    "Baseline": None,
    "RandomUnderSampler":     RandomUnderSampler(random_state=42),
    "TomekLinks":             TomekLinks(),
    "RandomOverSampler":      RandomOverSampler(random_state=42),
    "SMOTE":                  SMOTE(random_state=42),
    "ADASYN":                 ADASYN(random_state=42),
    "SMOTETomek":             SMOTETomek(random_state=42),
    "SMOTEENN":               SMOTEENN(random_state=42),
}
mejor_metodo_nombre = study.best_params['imbalance_method']
mejor_metodo = imbalance_methods[mejor_metodo_nombre]

mejor_xgb = xgb.XGBClassifier(
    n_estimators=study.best_params['n_estimators'],
    max_depth=study.best_params['max_depth'],
    learning_rate=study.best_params['learning_rate'],
    subsample=study.best_params['subsample'],
    colsample_bytree=study.best_params['colsample_bytree'],
    objective='multi:softmax',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

pasos_finales = [
    ('capping_outliers', IQRCapper(factor=1.5)),
    ('imputacion', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
]
if mejor_metodo is not None:
    pasos_finales.append(('sampler', mejor_metodo))

pasos_finales.append(('clf', mejor_xgb))
pipeline_final = Pipeline(pasos_finales)

pipeline_final.fit(X_train, y_train)

joblib.dump(pipeline_final, 'modelo_xgboost_ganador.pkl')
print("¡Modelo final entrenado y guardado correctamente como 'modelo_xgboost_ganador.pkl'!")

Reconstruyendo y entrenando el modelo final con toda la data...
¡Modelo final entrenado y guardado correctamente como 'modelo_xgboost_ganador.pkl'!


## STACKING CLASIFFIER

In [9]:
"""
modelos_base = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('xgb', xgb.XGBClassifier( 
        objective='multi:softprob', 
        num_class=3, 
        eval_metric='mlogloss',
        random_state=42
    ))
]

meta_modelo = LogisticRegression(max_iter=1000)

stacking_clf = StackingClassifier(
    estimators=modelos_base,
    final_estimator=meta_modelo,
    cv=5, 
    n_jobs=-1
)

mi_pipeline_stacking = Pipeline(steps=[
    ('capping_outliers', IQRCapper(factor=1.5)),
    ('imputacion', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()), 
    ('stacking', stacking_clf)
])


mi_pipeline_stacking.fit(X_train, np.ravel(y_train)) 

predicciones = mi_pipeline_stacking.predict(X_test)
probabilidades = mi_pipeline_stacking.predict_proba(X_test)

y_test1d = np.ravel(y_test)




exactitud = accuracy_score(y_test1d, predicciones)
print(f"Accuracy Global: {exactitud:.4f}")

# Calculamos F1-Score (usamos 'weighted' para tener en cuenta el desbalanceo)
f1_ponderado = f1_score(y_test1d, predicciones, average='weighted')
print(f"F1-Score (Weighted): {f1_ponderado:.4f}")


"""


'\nmodelos_base = [\n    (\'rf\', RandomForestClassifier(n_estimators=100, random_state=42)),\n    (\'knn\', KNeighborsClassifier(n_neighbors=5)),\n    (\'xgb\', xgb.XGBClassifier( \n        objective=\'multi:softprob\', \n        num_class=3, \n        eval_metric=\'mlogloss\',\n        random_state=42\n    ))\n]\n\nmeta_modelo = LogisticRegression(max_iter=1000)\n\nstacking_clf = StackingClassifier(\n    estimators=modelos_base,\n    final_estimator=meta_modelo,\n    cv=5, \n    n_jobs=-1\n)\n\nmi_pipeline_stacking = Pipeline(steps=[\n    (\'capping_outliers\', IQRCapper(factor=1.5)),\n    (\'imputacion\', SimpleImputer(strategy=\'mean\')),\n    (\'scaler\', StandardScaler()), \n    (\'stacking\', stacking_clf)\n])\n\n\nmi_pipeline_stacking.fit(X_train, np.ravel(y_train)) \n\npredicciones = mi_pipeline_stacking.predict(X_test)\nprobabilidades = mi_pipeline_stacking.predict_proba(X_test)\n\ny_test1d = np.ravel(y_test)\n\n\n\n\nexactitud = accuracy_score(y_test1d, predicciones)\nprint(